# Claim-Grounded Causal Hallucination Detection and Mitigation in Medical VLMs
### KG-free Cross-Modal Self-Consistency Framework (KG-LESS)

This notebook clones, configures, and runs the research implementation of the KG-LESS framework on **Dual T4 GPUs** in Kaggle, using the **HEAL-MedVQA** dataset from Hugging Face.

## 1. Setup and Environment Installation
We install the required PyTorch Geometric libraries and Hugging Face dependencies. Kaggle T4 environment typically comes with PyTorch pre-installed.

In [ ]:
!pip install torch-geometric
!pip install transformers datasets pyyaml scikit-learn pillow

## 2. Clone the Research Implementation Repository

In [ ]:
# Clean up existing folders and clone fresh
!rm -rf Hallucination-EMNLP
!git clone https://github.com/FaezehMillerAI/Hallucination-EMNLP.git
%cd Hallucination-EMNLP

## 3. Configure Hardware Settings
We can inspect the available T4 GPUs and configure `model_config.yaml` to run either in full mode (using `Qwen/Qwen2-VL-2B-Instruct`) or testing mode. Since Kaggle has 2x T4 GPUs, the pipeline automatically detects them and runs VLM operations on GPU 0 and GNN HGT operations on GPU 1.

In [ ]:
import torch
print("Available GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## 4. Run the Pipeline
Executes: 
1. Claim parsing and atomic fact decomposition.
2. PyG HeteroData Graph extraction (Visual patch nodes, token nodes, layerwise latent states).
3. Multi-task training of the HGT GNN detector (predicting hallucination, causes, sufficiency, faithfulness, localization).
4. Attention hook reallocation (mitigation) and re-decoding from hallucination onset.
5. F1, AUROC, and Pointing IoU evaluations.

In [ ]:
!python run_pipeline.py

## 5. Review Explainability Outputs

In [ ]:
import json
import glob

# List and display the first generated explanation report
reports = glob.glob("results/explanations/*.json")
if reports:
    with open(reports[0], 'r') as f:
        data = json.load(f)
        print(json.dumps(data, indent=2))
else:
    print("No reports found. Make sure the pipeline completed successfully.")